# Cross-product against relatedness — ggplot version

The same figures as `11_crossproduct_curves.ipynb`, drawn with ggplot2. This is
a **rendering** notebook: it recomputes nothing.

- bin means, pair counts and per-bin delete-block jackknife SEs come from
  `curves_<run>.tsv`, which `11_crossproduct_curves.ipynb` exports
- every fitted line is reconstructed **from the coefficients** in
  `estimators_<run>.tsv`, which `09_estimators.ipynb` writes from
  `src/he_estimators.py`
- the ranges, break points and thresholds come from `curve_params_<run>.tsv`

Nothing here re-derives a bin mean, a jackknife SE or a fitted coefficient, so
the two versions of the figure cannot disagree about the numbers — only about
how they are drawn. That is deliberate: nb09 used to carry its own copy of the
estimators and drifted from the module, and a second plotting notebook is an
easy way to reintroduce exactly that problem.

The reconstruction uses the identity documented in the module: `U` and `R`
share no bins, so the unrelated-range slope of the two-slope models is exactly
`h2_Unrel`.

**Prerequisites:** run `08_accumulate.ipynb`, then `09_estimators.ipynb`, then
the export cell at the end of `11_crossproduct_curves.ipynb`.

## Setup

In [ ]:
suppressPackageStartupMessages({library(readr); library(dplyr); library(tidyr)
  library(ggplot2); library(patchwork)})

# where 11_crossproduct_curves.ipynb and 09_estimators.ipynb wrote their tables
RUN <- "1kg_eur"
WB  <- path.expand("~/workspace/Data from All of Us Controlled Tier /shared-env-pilot")
BASE <- file.path(WB, "phenotypic_covariance_v9", RUN, "07_estimators")
OUT  <- Sys.getenv("CURVE_DIR", file.path(BASE, "crossproduct_curves"))
EST  <- Sys.getenv("EST_DIR",   file.path(BASE, "he_regression"))
need <- c(
  file.path(OUT, paste0("curves_", RUN, ".tsv")),
  file.path(OUT, paste0("curve_params_", RUN, ".tsv")),
  file.path(EST, paste0("estimators_", RUN, ".tsv")))
missing <- need[!file.exists(need)]
if (length(missing)) {
  stop(paste0(
    "missing input(s):\n  ", paste(missing, collapse = "\n  "), "\n\n",
    "This notebook only draws; it does not compute. Produce them first:\n",
    "    all three                          <- 09_estimators.ipynb\n",
    "which must have been run AFTER 08_accumulate.ipynb finished."), call. = FALSE)
}
cat("curves:", OUT, "\nestimators:", EST, "\nall three input tables present\n")

## Read the exported tables

In [ ]:
curves <- read_tsv(file.path(OUT, paste0("curves_", RUN, ".tsv")), show_col_types = FALSE)
P <- read_tsv(file.path(OUT, paste0("curve_params_", RUN, ".tsv")), show_col_types = FALSE) |>
  { \(d) setNames(as.list(d$value), d$key) }()
est <- read_tsv(file.path(EST, paste0("estimators_", RUN, ".tsv")), show_col_types = FALSE)

num <- function(k) as.numeric(P[[k]])
U_HI <- num("u_hi"); R_HI <- num("r_hi"); DEG_LO <- num("deg_lo")
## display settings -- this notebook's own, not inherited from the Python one.
## curves_*.tsv is unfiltered, so the sparse-bin cut belongs here.
MIN_PAIRS_PER_BIN <- 50
MAX_A <- R_HI                 # duplicate/MZ pairs are not part of this result
ZOOM_MIN <- -0.03; ZOOM <- 0.1
PO_XOFF <- 0.004              # nudge PO off FS so they do not overplot
curves <- dplyr::filter(curves, n_pairs >= MIN_PAIRS_PER_BIN, bin_mid <= MAX_A)
FS_RANGE <- as.numeric(strsplit(P$fs_range, ",")[[1]])
BANDS <- do.call(rbind, lapply(strsplit(strsplit(P$deg_bands, ";")[[1]], ":"), \(x) {
  lh <- as.numeric(strsplit(x[2], ",")[[1]]); data.frame(band = x[1], lo = lh[1], hi = lh[2]) }))
REF_TF <- P$ref_transform; REF_CS <- P$ref_covset

## Style

Matches the matplotlib version: `theme_bw` with a sparse major-only grid,
the same validated hues for pair class, the same ordered ramp for
residualisation model, and the same font fallback.

In [ ]:
## ── style: matches the matplotlib version ───────────────────────────────────
SURFACE <- "#fcfcfb"; INK <- "#0b0b0b"; INK2 <- "#52514e"; MUTED <- "#898781"; GRID <- "#e1e0d9"
CLASS_COL <- c(other = "#2a78d6", FS = "#eb6834", PO = "#1baf7a")
CLASS_LAB <- c(other = "unrelated / distant", FS = "full sibs", PO = "parent-offspring")
CLASS_SHP <- c(other = 16, FS = 15, PO = 17)
RAMP <- c("#86b6ef", "#3987e5", "#256abf", "#104281")
FONT <- {cand <- c("Helvetica Neue","Helvetica","Liberation Sans","Nimbus Sans","Arial")
         hit <- cand[cand %in% unique(systemfonts::system_fonts()$family)]
         if (length(hit)) hit[1] else ""}
theme_cov <- function(base = 10) {
  theme_bw(base_size = base, base_family = FONT) +
    theme(panel.background = element_rect(fill = SURFACE, colour = NA),
          plot.background = element_rect(fill = SURFACE, colour = NA),
          legend.background = element_blank(), legend.key = element_blank(),
          panel.border = element_rect(colour = INK2, linewidth = 0.4),
          panel.grid.major = element_line(colour = GRID, linewidth = 0.25),
          panel.grid.minor = element_blank(),          # way less grid
          axis.text = element_text(colour = INK2), axis.title = element_text(colour = INK2),
          plot.title = element_text(colour = INK, size = base), strip.background =
            element_rect(fill = "#f2f1ec", colour = INK2, linewidth = 0.4),
          strip.text = element_text(colour = INK2, size = base - 1.5),
          legend.title = element_text(size = base - 2), legend.text = element_text(size = base - 2.5))
}
YLAB <- expression(E * group("[", tilde(y)[i] * tilde(y)[j], "]"))

## Fitted lines from the coefficient table

`model_lines()` rebuilds each rung from its reported coefficients rather than
refitting. The offset models are only defined over `U` and `[DEG_LO, R_HI)`, so
they are drawn there and nowhere else — extrapolating past the last band edge
is what produced a spurious cliff in the matplotlib version.

In [ ]:
## ── fitted lines, reconstructed from the estimator table (no refitting) ─────
coefs <- function(phe, tf, cs) {
  e <- est |> filter(phenotype == phe, transform == tf, covset == cs)
  \(name, classes = "noPO") { v <- e$est[e$estimator == name & e$classes == classes]
                              if (length(v)) v[1] else NA_real_ }
}
band_of <- function(x) BANDS$band[vapply(x, \(v) { i <- which(v >= BANDS$lo & v < BANDS$hi)
                                                  if (length(i)) i[1] else NA_integer_ }, 1L)]
model_lines <- function(phe, tf, cs, xmax = MAX_A) {
  g <- coefs(phe, tf, cs); x <- seq(0, xmax, length.out = 600)
  off <- function(prefix, xs) {
    o <- vapply(band_of(xs), \(b) if (is.na(b)) NA_real_ else g(paste0(prefix, ".", b)), 1)
    ifelse(is.na(o), NA_real_, o)
  }
  bind_rows(
    data.frame(model = "one slope", x = x, y = g("h2_OneSlope") * x),
    data.frame(model = "two slopes", x = x,
               y = ifelse(x < U_HI, g("h2_Unrel") * x, g("h2_Rel") * x)),
    data.frame(model = "one slope + offsets", x = x,
               y = ifelse(x < U_HI, g("h2_OneSlopeOffsets") * x,
                          ifelse(x >= DEG_LO & x < R_HI,
                                 g("h2_OneSlopeOffsets") * x + off("OneSlopeOffsets", x), NA))),
    data.frame(model = "two slopes + offsets", x = x,
               y = ifelse(x < U_HI, g("h2_Unrel") * x,
                          ifelse(x >= DEG_LO & x < R_HI,
                                 g("h2_RelOffsets") * x + off("RelOffsets", x), NA))),
    data.frame(model = "FS range (noPO)", x = x,
               y = ifelse(x >= FS_RANGE[1] & x <= FS_RANGE[2],
                          g("b2_FS.slope") * x + g("b2_FS") / 2, NA))
  ) |> filter(!is.na(y))
}
LS <- c("one slope" = "dotted", "two slopes" = "dashed", "one slope + offsets" = "dotdash",
        "two slopes + offsets" = "longdash", "FS range (noPO)" = "twodash")

pts <- function(phe, tf, csets, xmax = MAX_A, xmin = -Inf) {
  curves |> filter(phenotype == phe, transform == tf, covset %in% csets,
                   class_set %in% names(CLASS_COL),
                   bin_mid <= xmax, bin_mid >= xmin) |>
    mutate(x = bin_mid + ifelse(class_set == "PO", PO_XOFF, 0),
           class_set = factor(class_set, levels = names(CLASS_COL)))
}

panel <- function(phe, tf, csets, xmax = MAX_A, xmin = -Inf, verbose = FALSE,
                  by_covset = TRUE) {
  d <- pts(phe, tf, csets, xmax, xmin)
  h2u <- coefs(phe, tf, csets[1])("h2_Unrel")
  p <- ggplot(d, aes(x = x, y = mean)) +
    geom_hline(yintercept = 0, colour = INK2, linewidth = 0.25) +
    geom_errorbar(aes(ymin = mean - se, ymax = mean + se,
                      colour = if (by_covset) covset else class_set),
                  width = 0, linewidth = 0.25, alpha = 0.3, na.rm = TRUE) +
    geom_abline(slope = h2u, intercept = 0, colour = INK, linewidth = 0.6) +
    geom_point(aes(shape = class_set, colour = if (by_covset) covset else class_set), size = 1.5) +
    scale_shape_manual(values = CLASS_SHP, labels = CLASS_LAB, name = "pair class") +
    labs(x = expression(a[ij]), y = YLAB) + theme_cov()
  p <- if (by_covset)
    p + scale_colour_manual(values = setNames(RAMP[seq_along(csets)], csets),
                            name = "residualisation model")
  else p + scale_colour_manual(values = CLASS_COL, labels = CLASS_LAB, guide = "none")
  if (verbose) {
    ml <- model_lines(phe, tf, csets[1], xmax)
    p <- p + geom_line(data = ml, aes(x = x, y = y, linetype = model),
                       colour = INK2, linewidth = 0.4, inherit.aes = FALSE) +
      scale_linetype_manual(values = LS, name = "regression model") +
      geom_text(data = d |> filter(class_set %in% c("FS", "PO")),
                aes(label = format(n_pairs, big.mark = ",")), size = 1.8,
                colour = MUTED, vjust = -0.9, show.legend = FALSE)
  }
  p
}

coef_panel <- function(phe, tf, csets) {
  # PO-only rows are omitted: a few hundred PO pairs give intervals wide
  # enough to set the axis scale by themselves. The contrast lives in
  # 09_estimators.ipynb's table.
  rows <- tribble(~estimator, ~classes, ~label, ~grp,
    "h2_Unrel","noPO","unrelated slope","slopes",
    "h2_OneSlopeOffsets","noPO","related slope (factor)","slopes",
    "b2_FS.slope","noPO","FS-range slope","slopes",
    "Unrel.intercept","noPO","unrelated intercept","intercepts")
  offs <- BANDS |> transmute(estimator = paste0("OneSlopeOffsets.", band),
                             classes = "noPO", label = paste("intercept", band),
                             grp = "intercepts")
  rows <- bind_rows(rows, offs,
    tribble(~estimator, ~classes, ~label, ~grp,
      "b2_FS","noPO","b2 (2x FS intercept)","intercepts"))
  d <- est |> filter(phenotype == phe, transform == tf, covset %in% csets) |>
    inner_join(rows, by = c("estimator", "classes")) |>
    mutate(label = factor(label, levels = rev(rows$label)),
           covset = factor(covset, levels = csets))
  mk <- function(g, unit) {
    dd <- filter(d, grp == g)
    p <- ggplot(dd, aes(est, label, colour = covset)) +
      geom_vline(xintercept = 0, colour = INK2, linewidth = 0.3) +
      geom_errorbar(aes(xmin = est - se, xmax = est + se), orientation = "y",
                    width = 0, linewidth = 0.3, alpha = 0.75, na.rm = TRUE) +
      geom_point(size = 1.4, na.rm = TRUE) +
      scale_colour_manual(values = setNames(RAMP[seq_along(csets)], csets), guide = "none") +
      labs(x = NULL, y = NULL, title = g) + theme_cov() +
      theme(panel.grid.major.y = element_blank())
    if (unit) p <- p + expand_limits(x = c(0, 1))
    p
  }
  mk("slopes", TRUE) / mk("intercepts", FALSE)
}

PHE <- sort(unique(curves$phenotype)); CS <- sort(unique(curves$covset))
CS <- c("base","base_pcs","base_pcs_zip3","base_pcs_zip3_ses") |> intersect(CS) |> { \(x) if (length(x)) x else CS }()

## Figure 1 — every phenotype

One facet per phenotype at the reference transform and covariate set, clean and
verbose.

In [ ]:
## Figure 1 — every phenotype, clean and verbose
for (vb in c(FALSE, TRUE)) {
  ps <- lapply(PHE, \(p) panel(p, REF_TF, REF_CS, verbose = vb, by_covset = FALSE) +
                        labs(title = p) + theme(legend.position = "none"))
  fig <- wrap_plots(ps, ncol = min(3, length(ps))) +
    plot_annotation(title = sprintf("Cross-product against relatedness - %s, %s, %s",
                                    RUN, REF_TF, REF_CS),
                    theme = theme_cov())
  f <- file.path(OUT, sprintf("ggplot_curves_all_%s_%s_%s.png", REF_TF, REF_CS,
                              if (vb) "verbose" else "clean"))
  ggsave(f, fig, width = 4.2 * min(3, length(ps)), height = 3.4 * ceiling(length(ps)/3),
         dpi = 200, bg = SURFACE)
  cat("->", f, "\n")
}

## Figure 2 — one phenotype, all residualisation models

Colour carries the residualisation model, shape the pair class. Zoom underneath,
coefficients to the right on their own scales.

In [ ]:
## Figure 2 — one phenotype, all residualisation models, zoom under, coefs right
DETAIL <- PHE[1]
for (vb in c(FALSE, TRUE)) {
  main <- panel(DETAIL, REF_TF, CS, verbose = vb) +
    labs(title = sprintf("%s - %s, all residualisation models", DETAIL, REF_TF)) +
    # legends below the panel: inside the axes they either sat on the data or
    # ran off the edge, and there are three of them when verbose
    # stacked, not side by side: three legends on one row get clipped
    theme(legend.position = "bottom", legend.box = "vertical",
          legend.margin = margin(0, 0, 0, 0), legend.spacing.y = unit(1, "pt")) +
    guides(linetype = guide_legend(nrow = 1, order = 3),
           shape = guide_legend(nrow = 1, order = 2),
           colour = guide_legend(nrow = 1, order = 1))
  zoom <- panel(DETAIL, REF_TF, CS, xmax = ZOOM, xmin = ZOOM_MIN, verbose = vb) +
    coord_cartesian(xlim = c(ZOOM_MIN, ZOOM)) +
    labs(title = bquote("zoom:" ~ .(ZOOM_MIN) <= a[ij] ~ "" <= .(ZOOM))) +
    theme(legend.position = "none")
  fig <- ((main / zoom) | coef_panel(DETAIL, REF_TF, CS)) + plot_layout(widths = c(3, 1.2))
  f <- file.path(OUT, sprintf("ggplot_curve_detail_%s_%s_%s.png", DETAIL, REF_TF,
                              if (vb) "verbose" else "clean"))
  ggsave(f, fig, width = 13, height = 9, dpi = 200, bg = SURFACE)
  cat("->", f, "\n")
}

## Copy notebook to bucket

In [ ]:
system(paste("gcloud storage cp",
             shQuote(path.expand("~/repos/AOU-covariance/notebooks/11r_crossproduct_curves.ipynb")),
             shQuote(paste0("gs://cloned-shared-env-pilot-wb-swift-sprout-7231/",
                            "phenotypic_covariance_v9/", RUN,
                            "/notebooks/11r_crossproduct_curves.ipynb"))))